# The tropical semiring, by hand

A *semiring* swaps the two operations of ordinary arithmetic. The **tropical**
(or min-plus / max-plus) semiring uses:

| | ordinary | tropical (min-plus) | tropical (max-plus) |
|---|---|---|---|
| "multiply" (⊗) | `a · b` | `a + b` | `a + b` |
| "add" (⊕) | `a + b` | `min(a, b)` | `max(a, b)` |

So a dot product `Σ_i a_i b_i` becomes `min_i (a_i + b_i)` (or `max_i`). No
multiplications — just additions and comparisons. This notebook builds intuition
for what that means for layers, gradients, and training.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# TikiTorch is research code, not on PyPI. In Colab, install from the repo.
# Replace REPO_URL with your TikiTorch checkout (it contains tikitorch/ + data/).
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/your-org/tikitorch.git"  # <-- set me
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(["git", "clone", REPO_URL, "tikitorch_repo"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo/tikitorch"], check=False)
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## Tropical linear = morphological erosion / dilation

`tropical_linear` is `y_j = min_i (W_ji + x_i)` (min-plus *erosion*);
`tropical_dilation` is the `max` dual. Each output picks **one** input — its
argmin/argmax — so the backward pass routes the gradient only through that single
winning index (a sparse, one-hot scatter). That sparsity is the source of both
the efficiency and the hardware-friendliness.


In [ ]:
import torch
import tikitorch.functional as F

x = torch.randn(1, 5, requires_grad=True)
W = torch.randn(3, 5)
y = F.tropical_linear(x, W)                 # (1,3): y_j = min_i (W[j,i] + x[i])
print("y =", y.detach().numpy().round(3))

y.sum().backward()
print("grad_x nonzeros:", (x.grad != 0).sum().item(), "of", x.numel(),
      "-> each output routed to ONE input (sparse gradient)")


## The soft → hard curriculum (Maslov dequantization)

The hard `min`/`max` has sparse gradients, which can be hard to train from
scratch. TikiTorch provides a **soft relaxation** controlled by a temperature
`tau`: `softmin(z, tau) = -tau · logsumexp(-z/tau)`, which is smooth (dense
gradients) for `tau > 0` and **converges to the hard min as `tau → 0`**. You
train soft, anneal `tau → 0`, and deploy the hard, multiplier-free op — the *same
weights* drive both. Let's watch a soft min converge to the hard min.


In [ ]:
import torch, tikitorch.functional as F
z = torch.tensor([[2.0, -1.0, 0.5, 3.0]])
hard = z.min(dim=-1).values.item()
for tau in [2.0, 1.0, 0.5, 0.1, 0.01]:
    soft = F.tropical_softmin(z, tau=tau, dim=-1).item()
    print(f"tau={tau:>4}: softmin={soft:+.4f}   (hard min={hard:+.4f})")


## Tropical matmul and the morphological view

`tropical_matmul(A, B)[i,j] = min_k (A[i,k] + B[k,j])`. The tropical identity is
a matrix with `0` on the diagonal and `+inf` off it; multiplying by it leaves `A`
unchanged — the same role the identity plays in ordinary linear algebra.

We can also *see* the morphology: a 1-D max-plus convolution (dilation) of a
spiky signal with a small structuring element spreads peaks; min-plus (erosion)
carves valleys. This is classical grayscale mathematical morphology — and it is
exactly what a tropical conv layer computes.


In [ ]:
import torch, tikitorch.functional as F

A = torch.randn(3, 3)
eye = torch.full((3, 3), float("inf")); eye.fill_diagonal_(0.0)
print("tropical_matmul(A, I) == A:", torch.allclose(F.tropical_matmul(A, eye), A))

# 1-D morphology: dilation (max-plus) vs erosion (min-plus) of a signal
import tikitorch.nn as nn
sig = torch.zeros(1, 1, 32); sig[0, 0, [8, 20]] = 1.0   # two spikes
dil = nn.MorphConv1d(1, kernel_size=5, mode="dilation", use_soft=False)
ero = nn.MorphConv1d(1, kernel_size=5, mode="erosion",  use_soft=False)
with torch.no_grad():
    dil.weight.zero_(); ero.weight.zero_()              # flat structuring element
    print("input peaks at:", (sig[0,0] > 0).nonzero().flatten().tolist())
    print("dilation spreads each peak into a plateau (max-plus).")


## The closure penalty

Forcing the whole loop into the tropical semiring with sign-only updates costs
some accuracy versus an unconstrained `AdamW + softmax-CE + FP32` baseline — the
**closure penalty**. On a hard char-LM the penalty decomposes (measured) as:
SignSGD vs AdamW ≈ −13pp, perceptron loss vs cross-entropy ≈ −17pp, INT16
quantization ≈ −1pp, hard tropical ops ≈ −5pp. The point of TikiTorch is not to
beat a GPU transformer on perplexity — it is to make the *whole loop closed* and
to give you the levers (the `tau` curriculum, the loss, the optimizer) to climb
the penalty back. Notebook **04** shows one lever recovering ~11pp,
multiplier-free.
